### Getting Setup (On Google Colab)

* Begin by installing some pip packages and the java development kit.

In [1]:
!pip install pyspark --quiet
#!pip install -U -q PyDrive --quiet
#!apt install openjdk-8-jdk-headless &> /dev/null

In [2]:
!apt-get update -y
!apt-get install -y openjdk-11-jdk-headless

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk-headless is already the newest version (11.0.30+7-1ubuntu1~22.04).
0 upgraded, 

* Then set the java environmental variable

RDD -- Resilient Distributed Datasets

In [4]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(os.environ["JAVA_HOME"])

/usr/lib/jvm/java-11-openjdk-amd64


In [5]:
!pip uninstall -y pyspark
!pip install pyspark==3.5.1

Found existing installation: pyspark 3.5.1
Uninstalling pyspark-3.5.1:
  Successfully uninstalled pyspark-3.5.1
  Using cached pyspark-3.5.1-py2.py3-none-any.whl
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


* Then connect to a SparkSession, setting the spark ui port to `4050`.

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



* Then we need to install ngrok which will allow us to place our local spark ui on the web.

In [ ]:
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip &> /dev/null
!unzip ngrok-stable-linux-amd64.zip &> /dev/null
get_ipython().system_raw('./ngrok http 4050 &')

* And finally we get a link our Spark UI

In [ ]:
!curl -s http://localhost:4040/api/tunnels | python3 -c \
    "import sys, json; print(json.load(sys.stdin)['tunnels'][0]['public_url'])"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
IndexError: list index out of range


### Looking Under the Hood

Now  let's again create an RDD from our movie records.

In [7]:
movies = ['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [11]:
sc = spark.sparkContext

In [12]:
movies_rdd = sc.parallelize(movies)
movies_rdd

ParallelCollectionRDD[4] at readRDDFromFile at PythonRDD.scala:289

And then let's capitalize the movies, and select the movies that begin with `d`.

In [13]:
movies_rdd.collect() #action

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [14]:
movies_rdd.take(3) #actions


['dark knight', 'dunkirk', 'pulp fiction']

In [15]:
movies[0].title()

'Dark Knight'

In [16]:
movies

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [17]:
transform=lambda i:i.title()
movies_title=[transform(i) for i in movies]

In [18]:
movies_title

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

In [19]:
movies_title_rdd=movies_rdd.map(transform) ## transformation

In [20]:
movies_title_rdd.collect()

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

In [21]:
movies_rdd.collect()

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [22]:
movies_rdd.filter(lambda movies : movies[0]=='d').collect()

['dark knight', 'dunkirk']

In [23]:
movies_rdd.map(lambda movie: movie.title()).take(2)
#transformations -- lazy transformations
## once you apply a transformation only the function is created but it is not applied
## you need an action to apply the transformation across your rdd

['Dark Knight', 'Dunkirk']

In [25]:
rdd1=movies_rdd.map(lambda movies :movies.title()).collect()

In [26]:
rdd2=movies_rdd.map(lambda movies : movies.title())

In [27]:
type(rdd2)

pyspark.rdd.PipelinedRDD

In [28]:
movies_rdd.map(lambda movies :movies.title()).collect()

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

Now as we know, Spark will partition the dataset across the cores of the executors, and then map through the records in parallel, returning all of the results.

> <img src="https://github.com/jigsawlabs-student/pyspark-rdds/blob/main/parallel.png?raw=1" width="60%">

Now let's change the function so that this time, instead of returning all of the results, we just return the first result.

In [29]:
movies_rdd.map(lambda movie: movie.title()).take(1)


['Dark Knight']

Now if we think about, this previous step, here we would not have to map through all of the steps just to return a single result.  And it turns out if we look at Spark, we can see that even though the dataset was distributed -- it only needed to perform work on a single partition to return one result.

> <img src="https://github.com/jigsawlabs-student/pyspark-rdds/blob/main/individual_task.png?raw=1" width="80%">

This ability, to see the end result that needs to be returned, and to work efficiently to only take the needed steps to return those results, is a valuable feature when working with large datasets.  And we can better see how Spark accomplishes it in the next section.

### A little experiment

If we run the code below, notice that nothing is returned.

In [30]:
movies_rdd.map(lambda movie: movie.title())

PythonRDD[13] at RDD at PythonRDD.scala:53

And even if we chain the map and the filter methods, still nothing is returned.

In [31]:
movies_rdd.map(lambda movie: movie.title()).filter(lambda movie: movie[0] == 'D').collect()

['Dark Knight', 'Dunkirk']

It's only when we add a collect function on the end, will some data be returned.

In [32]:
movies_rdd.filter(lambda movie: movie[0] == 'd').map(lambda movie: movie.title()).collect()

['Dark Knight', 'Dunkirk']

So above, nothing was returned when we ran the `map` and `collect` functions, because when we only executed those functions, Spark did not actually act on the data.  Then in the third line we finally did act on the data.  We told Spark that we want to both transform, and filter the data, and then return all of the results.  

So it's only when we called the `collect` function that Spark's driver determined the tasks to then send off to the executors and return the results.

### Transformations and Actions

So above we can see that the functions `map` and `filter` do not actually perform any work on our data.  Instead steps are only kicked off when we call the `collect` method.  

In Spark, the methods that kick off tasks and return results are called **actions** (eg. collect).  And methods like `map` and `transform` that do not are called **transformations**.  

1. Transformations

So we already saw that transformations include `map` and `filter`, and our transformations do not actually return results to our users.  Here's a couple other transformations.

* sample

The `sample` method allows us to take a random sample from our dataset.  

In [33]:
movies_rdd.sample(fraction = 0.5, withReplacement = False).collect()

[]

> Notice that it does not return any data.

* distinct

In [34]:
movies_rdd.distinct().collect()

['dark knight', 'avatar', 'dunkirk', 'pulp fiction']

> Distinct finds the unique results.  Notice that it also does not return data.

Finally, we have already seen `map`, which provides a one to one transformation of our records, and `select` which filters our data.  In each case, our transformations do not return data to us.

2. Actions

Actions are a bit more about the end result.  So far we've learned about `collect`, which returns *all* of the results of a series of transformations.  

* Take

We've also seen `take`, which limits our results to a subset.

In [35]:
movies_rdd.distinct().take(2)

['dark knight', 'avatar']

> So `take` is similar to the `LIMIT` function in SQL. Notice that here our records are returned.

* Count

In [36]:
movies_rdd.distinct().count()

4

Count simply counts the results.